# MiniOneRec：constrained RQ-KMeans SID construction

本 notebook 使用仓库的 `rq/rqkmeans_constrained.py`，为每个商品 embedding 生成三层 Semantic ID。该步骤使用 CPU，不需要 GPU。

输入：`Industrial_and_Scientific.emb-qwen-td.npy`。  
输出：`codebooks_constrained.npz`、`codes_constrained.npy`、`index.json`。

## 初始化：基础param、import包

In [1]:
# 配置项目、预处理数据和 constrained RQ-KMeans 脚本路径。
from pathlib import Path
import os
import subprocess
import sys

PROJECT_ROOT = Path(r'D:/转码ing/MiniOneRec-main')
DATASET = 'Industrial_and_Scientific'
DATA_DIR = PROJECT_ROOT / 'data' / 'Amazon18_2016_10_2018_11' / DATASET
RQ_DIR = PROJECT_ROOT / 'rq'
RQKMEANS_SCRIPT = RQ_DIR / 'rqkmeans_constrained.py'
EMBEDDING_FILE = DATA_DIR / f'{DATASET}.emb-qwen-td.npy'

# K 是每一层 codebook 的码数，L 是 SID 层数。
K = 256
L = 3
MAX_ITER = 100
SEED = 42

# False 时若已存在 SID 输出文件则停止，避免误覆盖已有结果。
ALLOW_OVERWRITE = False

In [2]:
import sys

!{sys.executable} -m pip install polars --only-binary=:all: --no-input --timeout 30 --retries 1 -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ---------------------------------------- 0.0/847.1 kB ? eta -:--:--
     ---------------------------------------- 0.0/847.1 kB ? eta -:--:--
     ------------ --------------------------- 262.1/847.1 kB ? eta -:--:--
     ---------------------- ------------- 524.3/847.1 kB 837.5 kB/s eta 0:00:01
     -------------------------------------- 847.1/847.1 kB 1.0 MB/s eta 0:00:00
     ---------------------------------------- 0.0/52.6 MB ? eta -:--:--
     ---------------------------------------- 0.0/52.6 MB ? eta -:--:--
     ---------------------------------------- 0.3/52.6 MB ? eta -:--:--
     ---------------------------------------- 0.5/52.6 MB 1.1 MB/s eta 0:00:47
      --------------------------------------- 0.8/52.6 MB 1.1 MB/s eta 0:00:48
      --------------------------------------- 0.8/52.6 MB 1.1 MB/s eta 0:00:48
      --------------------------------------- 1.0/52.6 MB 1.0 MB/s eta 0:00:50
     - -------------------

In [3]:
import polars

print("polars:", polars.__version__)

polars: 1.43.2


In [4]:
# 使用清华镜像实时安装 k-means-constrained。
import sys

!{sys.executable} -m pip install k-means-constrained --no-input --timeout 30 --retries 1 --disable-pip-version-check -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ---------------------------------------- 0.0/23.9 MB ? eta -:--:--
     ---------------------------------------- 0.3/23.9 MB ? eta -:--:--
     ---------------------------------------- 0.3/23.9 MB ? eta -:--:--
      -------------------------------------- 0.5/23.9 MB 882.6 kB/s eta 0:00:27
     - ------------------------------------- 0.8/23.9 MB 838.9 kB/s eta 0:00:28
     - ------------------------------------- 0.8/23.9 MB 838.9 kB/s eta 0:00:28
     - ------------------------------------- 1.0/23.9 MB 898.8 kB/s eta 0:00:26
     -- ------------------------------------ 1.3/23.9 MB 882.6 kB/s eta 0:00:26
     -- ------------------------------------ 1.6/23.9 MB 892.3 kB/s eta 0:00:25
     -- ------------------------------------ 1.6/23.9 MB 892.3 kB/s eta 0:00:25
     --- ----------------------------------- 2.1/23.9 MB 954.7 kB/s eta 0:00:23
     --- ----------------------------------- 2.1/23.9 MB 954.7 kB/s eta 0:00:23
   

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.4.6 which is incompatible.
tensorflow-cpu 2.18.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatible.


In [5]:
import k_means_constrained

print(
    "k-means-constrained:",
    getattr(k_means_constrained, "__version__", "installed"),
)

k-means-constrained: 0.9.1


In [6]:
# 检查 embedding、对应的 item 文件以及输出覆盖风险。
import json
import numpy as np

ITEM_FILE = DATA_DIR / f'{DATASET}.item.json'
OUTPUT_FILES = [
    DATA_DIR / f'{DATASET}.codebooks_constrained.npz',
    DATA_DIR / f'{DATASET}.codes_constrained.npy',
    DATA_DIR / f'{DATASET}.index.json',
]

required_files = [RQKMEANS_SCRIPT, EMBEDDING_FILE, ITEM_FILE]
missing_files = [path for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError('缺少以下文件：\n' + '\n'.join(map(str, missing_files)))

existing_files = [path for path in OUTPUT_FILES if path.exists()]
if existing_files and not ALLOW_OVERWRITE:
    raise FileExistsError(
        'SID 输出已存在。请先检查已有结果；确认需要重新生成后，将 ALLOW_OVERWRITE 改为 True：\n'
        + '\n'.join(map(str, existing_files))
    )

embeddings = np.load(EMBEDDING_FILE, mmap_mode='r')
with ITEM_FILE.open('r', encoding='utf-8') as file:
    items = json.load(file)

if embeddings.ndim != 2:
    raise ValueError(f'embedding 应为二维数组，实际 shape：{embeddings.shape}')
if len(items) != embeddings.shape[0]:
    raise ValueError(f'item 数 ({len(items)}) 与 embedding 行数 ({embeddings.shape[0]}) 不一致')
if K > embeddings.shape[0]:
    raise ValueError(f'K={K} 不能大于商品数 {embeddings.shape[0]}')

print('输入检查通过。')
print(f'Embedding: {EMBEDDING_FILE}')
print(f'Embedding shape: {embeddings.shape}, dtype: {embeddings.dtype}')
print(f'Item 数量: {len(items):,}')
print(f'SID 设置: {L} 层，每层 {K} 个码；理论路径容量：{K ** L:,}')
print(f'输出目录: {DATA_DIR}')


输入检查通过。
Embedding: D:\转码ing\MiniOneRec-main\data\Amazon18_2016_10_2018_11\Industrial_and_Scientific\Industrial_and_Scientific.emb-qwen-td.npy
Embedding shape: (3106, 1536), dtype: float32
Item 数量: 3,106
SID 设置: 3 层，每层 256 个码；理论路径容量：16,777,216
输出目录: D:\转码ing\MiniOneRec-main\data\Amazon18_2016_10_2018_11\Industrial_and_Scientific


## SID 生成

In [9]:
# 实时运行 constrained RQ-KMeans，并在失败时保留完整错误日志。
import os
import subprocess
import sys

command = [
    sys.executable,
    str(RQKMEANS_SCRIPT),
    "--dataset",
    DATASET,
    "--root",
    str(DATA_DIR),
    "--k",
    str(K),
    "--l",
    str(L),
    "--max_iter",
    str(MAX_ITER),
    "--seed",
    str(SEED),
    "--verbose",
]

print("将执行命令：")
print(" ".join(command))

run_env = os.environ.copy()
run_env["PYTHONUNBUFFERED"] = "1"

process = subprocess.Popen(
    command,
    cwd=RQ_DIR,
    env=run_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

exit_code = process.wait()

if exit_code != 0:
    raise RuntimeError(
        f"\nRQ-KMeans 脚本运行失败，退出码：{exit_code}。"
        "\n请复制上方最早出现的 Traceback 或 Error 内容。"
    )

print("\nRQ-KMeans 运行完成。")

将执行命令：
c:\Users\k\.conda\envs\tf_env\python.exe D:\转码ing\MiniOneRec-main\rq\rqkmeans_constrained.py --dataset Industrial_and_Scientific --root D:\转码ing\MiniOneRec-main\data\Amazon18_2016_10_2018_11\Industrial_and_Scientific --k 256 --l 3 --max_iter 100 --seed 42 --verbose
RQ-KMeans Constrained Training
root:  D:\转码ing\MiniOneRec-main\data\Amazon18_2016_10_2018_11\Industrial_and_Scientific
dataset:  Industrial_and_Scientific
Loaded embeddings from: D:\转码ing\MiniOneRec-main\data\Amazon18_2016_10_2018_11\Industrial_and_Scientific\Industrial_and_Scientific.emb-qwen-td.npy
Shape: (3106, 1536)
[Time: 0.02s]


=== Level 1/3 | K=256 ===
  Residual MSE before clustering: 13.821928
    Starting constrained K-means with K=256, n=3106, d=1536
    Cluster size constraints: [11, 13]
Initialization complete
Iteration  0, inertia 2696811.500
Iteration  1, inertia 1668383.500
Iteration  2, inertia 1618548.000
Iteration  3, inertia 1599916.250
Iteration  4, inertia 1588441.125
Iteration  5, inertia 1584

In [10]:
# 检查 index 映射是否覆盖全部商品，并验证去重后的完整 SID 不冲突。
from collections import Counter

INDEX_FILE = DATA_DIR / f'{DATASET}.index.json'
CODES_FILE = DATA_DIR / f'{DATASET}.codes_constrained.npy'
CODEBOOK_FILE = DATA_DIR / f'{DATASET}.codebooks_constrained.npz'

for path in [INDEX_FILE, CODES_FILE, CODEBOOK_FILE]:
    if not path.is_file():
        raise FileNotFoundError(f'未找到预期输出：{path}')

with INDEX_FILE.open('r', encoding='utf-8') as file:
    item_to_sid = json.load(file)
codes = np.load(CODES_FILE)
codebooks = np.load(CODEBOOK_FILE)

sid_strings = [''.join(tokens) for tokens in item_to_sid.values()]
sid_counts = Counter(sid_strings)
duplicate_sids = {sid: count for sid, count in sid_counts.items() if count > 1}
sid_lengths = Counter(len(tokens) for tokens in item_to_sid.values())

expected_ids = {str(i) for i in range(len(items))}
actual_ids = set(item_to_sid)

print('=== SID 输出检查 ===')
print(f'index 映射数: {len(item_to_sid):,}')
print(f'codes shape: {codes.shape}, dtype: {codes.dtype}')
print(f'codebook 名称: {list(codebooks.files)}')
print(f'完整 SID 唯一数: {len(sid_counts):,}')
print(f'完整 SID 冲突数: {len(duplicate_sids):,}')
print(f'SID token 长度分布: {dict(sorted(sid_lengths.items()))}')
print(f'缺失 item_id 数: {len(expected_ids - actual_ids):,}')
print(f'多余 item_id 数: {len(actual_ids - expected_ids):,}')

print('\n=== 前 5 个 item_id → SID ===')
for item_id in list(item_to_sid)[:5]:
    print({
        'item_id': item_id,
        'sid_tokens': item_to_sid[item_id],
        'sid_text': ''.join(item_to_sid[item_id]),
        'title': items[item_id].get('title', '')[:100],
    })

if duplicate_sids or expected_ids != actual_ids:
    raise RuntimeError('SID 映射校验未通过，请先检查上述统计后再进入 convert_dataset.py。')

print('\nSID 映射校验通过：可以进入 convert_dataset.py。')

=== SID 输出检查 ===
index 映射数: 3,106
codes shape: (3106, 3), dtype: int32
codebook 名称: ['codebook_0', 'codebook_1', 'codebook_2']
完整 SID 唯一数: 3,106
完整 SID 冲突数: 0
SID token 长度分布: {3: 2657, 4: 449}
缺失 item_id 数: 0
多余 item_id 数: 0

=== 前 5 个 item_id → SID ===
{'item_id': '0', 'sid_tokens': ['<a_156>', '<b_97>', '<c_225>'], 'sid_text': '<a_156><b_97><c_225>', 'title': 'SUPCO SPP6 Relay/Capacitor Hard Start Kit with 500% Increase Starting Torque'}
{'item_id': '1', 'sid_tokens': ['<a_17>', '<b_245>', '<c_71>'], 'sid_text': '<a_17><b_245><c_71>', 'title': 'Stanley TRA708T Sharpshooter 1/2-Inch Leg Length Staples, Steel (1000 Count)'}
{'item_id': '2', 'sid_tokens': ['<a_2>', '<b_220>', '<c_205>', '<d_1>'], 'sid_text': '<a_2><b_220><c_205><d_1>', 'title': 'Kreg SML-C125-500 1-1/4-Inch #8 Coarse Pocket Hole Screws with Washer-Head, 500-Pack'}
{'item_id': '3', 'sid_tokens': ['<a_184>', '<b_151>', '<c_89>'], 'sid_text': '<a_184><b_151><c_89>', 'title': 'Dico 541-774-21/2 Nyalox Cup Brush 21/2-Inch Gr

## evaluation

In [11]:
# 评估 constrained RQ-KMeans 的量化质量、码本使用情况、分布均衡性和 SID 冲突情况。
from pathlib import Path
from collections import Counter
import json
import math
import numpy as np
import pandas as pd

embedding_file = DATA_DIR / f"{DATASET}.emb-qwen-td.npy"
codes_file = DATA_DIR / f"{DATASET}.codes_constrained.npy"
codebook_file = DATA_DIR / f"{DATASET}.codebooks_constrained.npz"
index_file = DATA_DIR / f"{DATASET}.index.json"

# 读取原始 embedding、每层离散 code、每层 codebook 以及去重后的最终 SID。
embeddings = np.load(embedding_file).astype(np.float32)
codes = np.load(codes_file)
codebooks_data = np.load(codebook_file)

with index_file.open("r", encoding="utf-8") as file:
    item_to_sid = json.load(file)

num_items, num_levels = codes.shape
embedding_dim = embeddings.shape[1]

if num_items != embeddings.shape[0]:
    raise ValueError(
        f"embedding 行数 ({embeddings.shape[0]}) 与 codes 行数 ({num_items}) 不一致。"
    )

# 按每层 codebook 重构 embedding：x_hat = C1[c1] + C2[c2] + ...
reconstruction = np.zeros_like(embeddings, dtype=np.float32)

for level in range(num_levels):
    codebook = codebooks_data[f"codebook_{level}"].astype(np.float32)
    reconstruction += codebook[codes[:, level]]

# 计算重构误差与原向量、重构向量之间的余弦相似度。
mse = float(np.mean((embeddings - reconstruction) ** 2))
rmse = float(np.sqrt(mse))
original_energy = float(np.mean(embeddings ** 2))
normalized_mse = float(mse / original_energy) if original_energy > 0 else float("nan")

original_norm = np.linalg.norm(embeddings, axis=1)
reconstruction_norm = np.linalg.norm(reconstruction, axis=1)
cosine_similarity = np.sum(embeddings * reconstruction, axis=1) / (
    original_norm * reconstruction_norm + 1e-12
)

# 统计每层 codebook 的实际使用数、利用率和熵。
level_metrics = []

for level in range(num_levels):
    level_codes = codes[:, level]
    codebook_size = codebooks_data[f"codebook_{level}"].shape[0]

    counts = np.bincount(level_codes, minlength=codebook_size)
    used_codes = int(np.count_nonzero(counts))
    utilization = used_codes / codebook_size

    probabilities = counts[counts > 0] / counts.sum()
    entropy = float(-np.sum(probabilities * np.log2(probabilities)))
    max_entropy = math.log2(codebook_size)
    normalized_entropy = entropy / max_entropy if max_entropy > 0 else float("nan")

    level_metrics.append(
        {
            "level": level + 1,
            "codebook_size": codebook_size,
            "used_codes": used_codes,
            "utilization": utilization,
            "min_count": int(counts.min()),
            "max_count": int(counts.max()),
            "mean_count": float(counts.mean()),
            "entropy_bits": entropy,
            "normalized_entropy": normalized_entropy,
        }
    )

level_metrics_df = pd.DataFrame(level_metrics)

# 原始 code 路径的冲突：尚未经过 deal_with_deduplicate 的结果。
raw_sid_strings = ["|".join(map(str, row)) for row in codes]
raw_sid_counts = Counter(raw_sid_strings)
raw_unique_paths = len(raw_sid_counts)
raw_collision_items = num_items - raw_unique_paths
raw_collision_rate = raw_collision_items / num_items

# 最终 SID 冲突：应当为 0，因为 constrained 脚本已对冲突路径追加去重 token。
final_sid_strings = ["".join(tokens) for tokens in item_to_sid.values()]
final_sid_counts = Counter(final_sid_strings)
final_unique_paths = len(final_sid_counts)
final_collision_items = len(final_sid_strings) - final_unique_paths
final_collision_rate = final_collision_items / len(final_sid_strings)

print("=== RQ-KMeans 总体指标 ===")
print(f"商品数: {num_items:,}")
print(f"Embedding 维度: {embedding_dim:,}")
print(f"SID 层数: {num_levels}")
print(f"Reconstruction MSE: {mse:.8f}")
print(f"Reconstruction RMSE: {rmse:.8f}")
print(f"Normalized MSE: {normalized_mse:.6f}")
print(f"Mean cosine similarity: {cosine_similarity.mean():.6f}")
print(f"Median cosine similarity: {np.median(cosine_similarity):.6f}")
print(f"Minimum cosine similarity: {cosine_similarity.min():.6f}")

print("\n=== 完整 SID 冲突情况 ===")
print(f"原始完整路径唯一数: {raw_unique_paths:,}")
print(f"原始冲突商品数: {raw_collision_items:,}")
print(f"原始 collision rate: {raw_collision_rate:.4%}")
print(f"最终 SID 映射数: {len(final_sid_strings):,}")
print(f"最终完整 SID 唯一数: {final_unique_paths:,}")
print(f"最终冲突商品数: {final_collision_items:,}")
print(f"最终 collision rate: {final_collision_rate:.4%}")

print("\n=== 每层 codebook 使用情况 ===")
display(
    level_metrics_df.style.format(
        {
            "utilization": "{:.2%}",
            "mean_count": "{:.2f}",
            "entropy_bits": "{:.4f}",
            "normalized_entropy": "{:.2%}",
        }
    )
)

print("\n=== 前 5 个原始 code 与最终 SID 样本 ===")
for item_id in list(item_to_sid)[:5]:
    item_index = int(item_id)
    print(
        {
            "item_id": item_id,
            "raw_codes": codes[item_index].tolist(),
            "final_sid": item_to_sid[item_id],
        }
    )

if final_collision_rate != 0:
    raise RuntimeError("最终 SID 仍有冲突，不能直接进入 convert_dataset.py。")

=== RQ-KMeans 总体指标 ===
商品数: 3,106
Embedding 维度: 1,536
SID 层数: 3
Reconstruction MSE: 0.17439200
Reconstruction RMSE: 0.41760268
Normalized MSE: 0.012617
Mean cosine similarity: 0.993515
Median cosine similarity: 0.993782
Minimum cosine similarity: 0.909835

=== 完整 SID 冲突情况 ===
原始完整路径唯一数: 2,839
原始冲突商品数: 267
原始 collision rate: 8.5963%
最终 SID 映射数: 3,106
最终完整 SID 唯一数: 3,106
最终冲突商品数: 0
最终 collision rate: 0.0000%

=== 每层 codebook 使用情况 ===


,level,codebook_size,used_codes,utilization,min_count,max_count,mean_count,entropy_bits,normalized_entropy
0,1,256,256,100.00%,11,13,12.13,7.9955,99.94%
1,2,256,256,100.00%,11,13,12.13,7.9958,99.95%
2,3,256,256,100.00%,11,13,12.13,7.9955,99.94%



=== 前 5 个原始 code 与最终 SID 样本 ===
{'item_id': '0', 'raw_codes': [155, 96, 224], 'final_sid': ['<a_156>', '<b_97>', '<c_225>']}
{'item_id': '1', 'raw_codes': [16, 244, 70], 'final_sid': ['<a_17>', '<b_245>', '<c_71>']}
{'item_id': '2', 'raw_codes': [1, 219, 204], 'final_sid': ['<a_2>', '<b_220>', '<c_205>', '<d_1>']}
{'item_id': '3', 'raw_codes': [183, 150, 88], 'final_sid': ['<a_184>', '<b_151>', '<c_89>']}
{'item_id': '4', 'raw_codes': [213, 3, 209], 'final_sid': ['<a_214>', '<b_4>', '<c_210>']}


ps：出现冲突的话给冲突项新增一个去重编号比如<d_1>

Final SID collision rate = 0%

Codebook utilization 接近 100%

Normalized entropy 接近 100%

Mean cosine similarity 越接近 1 越好

Normalized MSE 越小越好